In [ ]:
import os
import sys

sys.path.append("/home/justin/code/point-to-pose/")
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import glob

import torch
import torch.nn.functional as F

# Try importing sklearn for advanced descriptor visualization (optional)
try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("Note: sklearn not available. Some descriptor visualizations will be simplified.")

# LightGlue imports
from lightglue import LightGlue, SuperPoint, DISK, SIFT, ALIKED, DoGHardNet
from lightglue.utils import load_image, rbd
from lightglue import viz2d

# Check if CUDA is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


In [ ]:
# Configure data path
# data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/edamame_box"
# data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/green_bowl"
data_path = "/home/justin/data/HO3D_V3/evaluation/AP10"


In [ ]:
# Function to load RGB images (from pipeline_test.ipynb)
def load_rgb_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load RGB images from the /rgb subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.jpg', '.jpeg', '.png'])
    
    Returns:
        List[np.ndarray]: List of RGB images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.jpg', '.jpeg', '.png']
    
    rgb_folder = os.path.join(folder_path, 'rgb')
    if not os.path.exists(rgb_folder):
        raise FileNotFoundError(f"RGB folder not found: {rgb_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(rgb_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            img = cv2.imread(file_path)
            if img is not None:
                # Convert BGR to RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img_rgb)
    
    return images

# Function to load mask images (from pipeline_test.ipynb)
def load_mask_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load mask images from the /masks subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /masks subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.jpg', '.jpeg'])
    
    Returns:
        List[np.ndarray]: List of mask images as numpy arrays (binary or grayscale)
    """
    if file_extensions is None:
        file_extensions = ['.png', '.jpg', '.jpeg']
    
    masks_folder = os.path.join(folder_path, 'masks')
    if not os.path.exists(masks_folder):
        raise FileNotFoundError(f"Masks folder not found: {masks_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(masks_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load mask as grayscale
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    
    return images

# Helper function to extract masked region from image
def extract_masked_region(img: np.ndarray, mask: np.ndarray, 
                         padding: int = 10) -> Tuple[np.ndarray, np.ndarray, Tuple[int, int]]:
    """
    Extract the bounding box region containing the mask from an image.
    
    Args:
        img: Image as numpy array (H, W, 3) RGB
        mask: Binary mask (H, W)
        padding: Additional padding around the bounding box
        
    Returns:
        cropped_image: Cropped image containing the masked region
        cropped_mask: Cropped mask aligned with the image
        bbox: Bounding box (x_min, y_min, x_max, y_max) in original image coordinates
    """
    # Ensure mask is binary
    if mask.max() > 1:
        mask_binary = (mask > 128).astype(np.uint8)
    else:
        mask_binary = mask.astype(np.uint8)
    
    # Find bounding box
    coords = np.where(mask_binary > 0)
    if len(coords[0]) == 0:
        # No mask found, return original image
        return img, mask, (0, 0, img.shape[1], img.shape[0])
    
    y_min, y_max = max(0, coords[0].min() - padding), min(img.shape[0], coords[0].max() + padding + 1)
    x_min, x_max = max(0, coords[1].min() - padding), min(img.shape[1], coords[1].max() + padding + 1)
    
    # Crop image and mask
    cropped_img = img[y_min:y_max, x_min:x_max].copy()
    cropped_mask = mask_binary[y_min:y_max, x_min:x_max].copy()
    
    return cropped_img, cropped_mask, (x_min, y_min, x_max, y_max)

# Helper function to convert keypoints from cropped image coordinates to original image coordinates
def transform_keypoints_to_original(kpts: np.ndarray, bbox: Tuple[int, int, int, int]) -> np.ndarray:
    """
    Transform keypoints from cropped image coordinates to original image coordinates.
    
    Args:
        kpts: Keypoints in cropped image coordinates (N, 2)
        bbox: Bounding box (x_min, y_min, x_max, y_max)
        
    Returns:
        Transformed keypoints in original image coordinates
    """
    x_min, y_min = bbox[0], bbox[1]
    kpts_original = kpts.copy()
    kpts_original[:, 0] += x_min
    kpts_original[:, 1] += y_min
    return kpts_original

# Helper function to filter matches based on masks
def filter_matches_by_mask(points0: np.ndarray, points1: np.ndarray, 
                          mask0: np.ndarray, mask1: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Filter matches to keep only those where keypoints are within the mask regions.
    
    Args:
        points0: Keypoints in first image (K, 2)
        points1: Keypoints in second image (K, 2)
        mask0: Binary mask for first image (H, W)
        mask1: Binary mask for second image (H, W)
    
    Returns:
        Filtered points0 and points1 arrays
    """
    # Convert to numpy if needed
    if isinstance(points0, torch.Tensor):
        points0 = points0.cpu().numpy()
    if isinstance(points1, torch.Tensor):
        points1 = points1.cpu().numpy()
    
    # Ensure masks are binary (threshold at 128 if grayscale)
    if mask0.max() > 1:
        mask0_binary = (mask0 > 128).astype(np.uint8)
    else:
        mask0_binary = mask0.astype(np.uint8)
        
    if mask1.max() > 1:
        mask1_binary = (mask1 > 128).astype(np.uint8)
    else:
        mask1_binary = mask1.astype(np.uint8)
    
    # Check which keypoints are within mask regions
    valid_indices = []
    for i in range(len(points0)):
        x0, y0 = int(points0[i, 0]), int(points0[i, 1])
        x1, y1 = int(points1[i, 0]), int(points1[i, 1])
        
        # Check bounds and mask values
        h0, w0 = mask0_binary.shape
        h1, w1 = mask1_binary.shape
        
        if (0 <= x0 < w0 and 0 <= y0 < h0 and 
            0 <= x1 < w1 and 0 <= y1 < h1):
            if mask0_binary[y0, x0] > 0 and mask1_binary[y1, x1] > 0:
                valid_indices.append(i)
    
    if len(valid_indices) == 0:
        print("Warning: No matches found within mask regions!")
        return np.array([]).reshape(0, 2), np.array([]).reshape(0, 2)
    
    return points0[valid_indices], points1[valid_indices]

# Helper function to convert numpy RGB image to torch tensor for LightGlue
def numpy_to_lightglue_image(img: np.ndarray) -> torch.Tensor:
    """
    Convert numpy RGB image (H, W, 3) to torch tensor (3, H, W) normalized in [0, 1].
    
    Args:
        img: numpy array of shape (H, W, 3) with values in [0, 255]
    
    Returns:
        torch.Tensor: tensor of shape (3, H, W) with values in [0, 1]
    """
    # Convert to float and normalize to [0, 1]
    img_float = img.astype(np.float32) / 255.0
    # Convert from (H, W, 3) to (3, H, W)
    img_tensor = torch.from_numpy(img_float).permute(2, 0, 1)
    return img_tensor.to(device)


In [ ]:
# Load RGB images and masks
rgb_images = load_rgb_images(data_path)
print(f"Loaded {len(rgb_images)} RGB images")

try:
    mask_images = load_mask_images(data_path)
    print(f"Loaded {len(mask_images)} mask images")
    if len(mask_images) != len(rgb_images):
        print(f"Warning: Number of masks ({len(mask_images)}) doesn't match number of RGB images ({len(rgb_images)})")
except FileNotFoundError as e:
    print(f"Warning: {e}")
    mask_images = []

# Display first image and mask to verify loading
if len(rgb_images) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(rgb_images[0])
    axes[0].set_title("First RGB image")
    axes[0].axis('off')
    
    if len(mask_images) > 0:
        axes[1].imshow(mask_images[0], cmap='gray')
        axes[1].set_title("First mask")
        axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, "No masks loaded", ha='center', va='center')
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# Initialize SuperPoint extractor and LightGlue matcher
extractor = SuperPoint(max_num_keypoints=4096).eval().to(device)
matcher = LightGlue(features='superpoint').eval().to(device)

print("SuperPoint extractor and LightGlue matcher initialized")


In [ ]:
# Select two consecutive images for matching
if len(rgb_images) < 2:
    raise ValueError("Need at least 2 images for matching")

# Convert numpy images to torch tensors
image0_np = rgb_images[0]
image1_np = rgb_images[1]

image0 = numpy_to_lightglue_image(image0_np)
image1 = numpy_to_lightglue_image(image1_np)

print(f"Image 0 shape: {image0.shape}")
print(f"Image 1 shape: {image1.shape}")

# Display both images side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(image0_np)
axes[0].set_title("Image 0")
axes[0].axis('off')
axes[1].imshow(image1_np)
axes[1].set_title("Image 1")
axes[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Extract local features
print("Extracting features from image 0...")
feats0 = extractor.extract(image0)  # auto-resize the image, disable with resize=None
print("Extracting features from image 1...")
feats1 = extractor.extract(image1)

print(f"Features 0 keys: {feats0.keys()}")
print(f"Features 1 keys: {feats1.keys()}")
if 'keypoints' in feats0:
    kp_shape = feats0['keypoints'].shape
    print(f"Keypoints shape in image 0: {kp_shape}")
if 'keypoints' in feats1:
    kp_shape = feats1['keypoints'].shape
    print(f"Keypoints shape in image 1: {kp_shape}")
if 'keypoint_scores' in feats0:
    print(f"Keypoint scores shape in image 0: {feats0['keypoint_scores'].shape}")
if 'descriptors' in feats0:
    print(f"Descriptor shape in image 0: {feats0['descriptors'].shape}")


In [ ]:
# Match the features
print("Matching features...")
matches01 = matcher({'image0': feats0, 'image1': feats1})

# Remove batch dimension
feats0, feats1, matches01 = [rbd(x) for x in [feats0, feats1, matches01]]

# Extract matches
matches = matches01['matches']  # indices with shape (K,2)
points0 = feats0['keypoints'][matches[..., 0]]  # coordinates in image #0, shape (K,2)
points1 = feats1['keypoints'][matches[..., 1]]  # coordinates in image #1, shape (K,2)

print(f"Number of matches (before mask filtering): {len(matches)}")
print(f"Match confidence (if available): {matches01.get('matching_scores', 'N/A')}")

# Check if pruning information is available
has_pruning = 'prune0' in matches01 and 'prune1' in matches01
has_stop = 'stop' in matches01
if has_pruning:
    print("Pruning information available")
if has_stop:
    print(f"LightGlue stopped after {matches01['stop']} layers")

# Get masks if available (check if mask_images exists)
if 'mask_images' in globals() and len(mask_images) > 0:
    mask0 = mask_images[0] if len(mask_images) > 0 else None
    mask1 = mask_images[1] if len(mask_images) > 1 else None
else:
    mask0 = None
    mask1 = None
    if 'mask_images' not in globals():
        print("Note: mask_images not loaded. Run cell 3 to load masks.")

# Filter matches by mask if masks are available
if mask0 is not None and mask1 is not None:
    points0_filtered, points1_filtered = filter_matches_by_mask(points0, points1, mask0, mask1)
    print(f"Number of matches (after mask filtering): {len(points0_filtered)}")
    points0 = points0_filtered
    points1 = points1_filtered


In [ ]:
# Visualize extracted features
def visualize_keypoint_scores(img: np.ndarray, keypoints: torch.Tensor, scores: torch.Tensor, 
                              title: str = "Keypoint Scores"):
    """
    Visualize keypoint detection scores as a heatmap and scatter plot.
    
    Args:
        img: Image as numpy array (H, W, 3) RGB
        keypoints: Keypoints tensor (N, 2)
        scores: Keypoint scores tensor (N,)
        title: Plot title
    """
    # Convert to numpy
    if isinstance(keypoints, torch.Tensor):
        keypoints = keypoints.cpu().numpy()
    if isinstance(scores, torch.Tensor):
        scores = scores.cpu().numpy()
    
    # Create heatmap
    h, w = img.shape[:2]
    heatmap = np.zeros((h, w), dtype=np.float32)
    
    # Add scores to heatmap (using Gaussian-like distribution around each keypoint)
    for i in range(len(keypoints)):
        x, y = int(keypoints[i, 0]), int(keypoints[i, 1])
        if 0 <= x < w and 0 <= y < h:
            # Create a small Gaussian blob at the keypoint location
            radius = 5
            score = scores[i]
            for dy in range(-radius, radius+1):
                for dx in range(-radius, radius+1):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < w and 0 <= ny < h:
                        dist = np.sqrt(dx*dx + dy*dy)
                        if dist <= radius:
                            weight = np.exp(-dist**2 / (2 * (radius/2)**2))
                            heatmap[ny, nx] = max(heatmap[ny, nx], score * weight)
    
    # Normalize heatmap
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Heatmap
    im = axes[1].imshow(heatmap, cmap='hot', interpolation='bilinear')
    axes[1].set_title("Keypoint Score Heatmap")
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    
    # Overlay keypoints on image
    axes[2].imshow(img)
    # Color keypoints by score
    scatter = axes[2].scatter(keypoints[:, 0], keypoints[:, 1], 
                             c=scores, cmap='hot', s=30, alpha=0.7, 
                             edgecolors='white', linewidths=0.5)
    axes[2].set_title(f"{title} - {len(keypoints)} keypoints")
    axes[2].axis('off')
    plt.colorbar(scatter, ax=axes[2], fraction=0.046, label='Score')
    
    plt.tight_layout()
    plt.show()

# Visualize keypoint scores for both images
if 'keypoint_scores' in feats0 and 'keypoints' in feats0:
    # Remove batch dimension if present
    kpts0_viz = feats0['keypoints']
    scores0_viz = feats0['keypoint_scores']
    if kpts0_viz.dim() > 2:  # Has batch dimension
        kpts0_viz = kpts0_viz[0]
        scores0_viz = scores0_viz[0]
    
    visualize_keypoint_scores(image0_np, kpts0_viz, scores0_viz, 
                              title="Image 0 Keypoint Scores")

if 'keypoint_scores' in feats1 and 'keypoints' in feats1:
    # Remove batch dimension if present
    kpts1_viz = feats1['keypoints']
    scores1_viz = feats1['keypoint_scores']
    if kpts1_viz.dim() > 2:  # Has batch dimension
        kpts1_viz = kpts1_viz[0]
        scores1_viz = scores1_viz[0]
    
    visualize_keypoint_scores(image1_np, kpts1_viz, scores1_viz, 
                              title="Image 1 Keypoint Scores")


In [ ]:
# Simple descriptor statistics visualization (no sklearn needed)
def visualize_descriptor_stats(keypoints: torch.Tensor, descriptors: torch.Tensor,
                               img: np.ndarray, title: str = "Descriptor Statistics"):
    """
    Visualize descriptor statistics without dimensionality reduction.
    """
    # Convert to numpy
    if isinstance(keypoints, torch.Tensor):
        keypoints = keypoints.cpu().numpy()
    if isinstance(descriptors, torch.Tensor):
        descriptors = descriptors.cpu().numpy()
    
    # Remove batch dimension if present
    if descriptors.ndim > 2:
        descriptors = descriptors[0]
        keypoints = keypoints[0]
    
    # Transpose if needed
    if descriptors.shape[0] != len(keypoints):
        descriptors = descriptors.T
    
    # Compute descriptor statistics
    desc_mean = np.mean(descriptors, axis=1)  # Mean across descriptor dimensions
    desc_std = np.std(descriptors, axis=1)    # Std across descriptor dimensions
    desc_norm = np.linalg.norm(descriptors, axis=1)  # L2 norm
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Original image
    axes[0, 0].imshow(img)
    axes[0, 0].set_title("Original Image")
    axes[0, 0].axis('off')
    
    # Keypoints colored by descriptor mean
    axes[0, 1].imshow(img)
    scatter1 = axes[0, 1].scatter(keypoints[:, 0], keypoints[:, 1], 
                                  c=desc_mean, cmap='viridis', s=30, 
                                  alpha=0.7, edgecolors='white', linewidths=0.5)
    axes[0, 1].set_title('Keypoints by Descriptor Mean')
    axes[0, 1].axis('off')
    plt.colorbar(scatter1, ax=axes[0, 1], fraction=0.046)
    
    # Keypoints colored by descriptor norm
    axes[0, 2].imshow(img)
    scatter2 = axes[0, 2].scatter(keypoints[:, 0], keypoints[:, 1], 
                                  c=desc_norm, cmap='hot', s=30, 
                                  alpha=0.7, edgecolors='white', linewidths=0.5)
    axes[0, 2].set_title('Keypoints by Descriptor Norm')
    axes[0, 2].axis('off')
    plt.colorbar(scatter2, ax=axes[0, 2], fraction=0.046)
    
    # Histogram of descriptor means
    axes[1, 0].hist(desc_mean, bins=50, edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Descriptor Mean')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Descriptor Means')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Histogram of descriptor norms
    axes[1, 1].hist(desc_norm, bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1, 1].set_xlabel('Descriptor Norm')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of Descriptor Norms')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Descriptor statistics
    axes[1, 2].axis('off')
    stats_text = f"""
    Descriptor Statistics:
    Shape: {descriptors.shape}
    Mean: {desc_mean.mean():.4f} ± {desc_mean.std():.4f}
    Norm: {desc_norm.mean():.4f} ± {desc_norm.std():.4f}
    Std: {desc_std.mean():.4f} ± {desc_std.std():.4f}
    Keypoints: {len(keypoints)}
    """
    axes[1, 2].text(0.1, 0.5, stats_text, fontsize=12, 
                    verticalalignment='center', family='monospace')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize feature descriptors using dimensionality reduction
def visualize_descriptors(keypoints: torch.Tensor, descriptors: torch.Tensor, 
                         img: np.ndarray, n_components: int = 2, title: str = "Descriptor Visualization"):
    """
    Visualize feature descriptors by reducing dimensionality and coloring keypoints.
    
    Args:
        keypoints: Keypoints tensor (N, 2)
        descriptors: Descriptor tensor (N, D) where D is descriptor dimension
        img: Image as numpy array (H, W, 3) RGB
        n_components: Number of dimensions for reduction (2 or 3)
        title: Plot title
    """
    if not HAS_SKLEARN:
        print("sklearn not available. Using simple descriptor statistics visualization instead.")
        visualize_descriptor_stats(keypoints, descriptors, img, title)
        return
    
    # Convert to numpy
    if isinstance(keypoints, torch.Tensor):
        keypoints = keypoints.cpu().numpy()
    if isinstance(descriptors, torch.Tensor):
        descriptors = descriptors.cpu().numpy()
    
    # Remove batch dimension if present
    if descriptors.ndim > 2:
        descriptors = descriptors[0]
        keypoints = keypoints[0]
    
    print(f"Descriptor shape: {descriptors.shape}")
    print(f"Reducing {descriptors.shape[1]} dimensions to {n_components}...")
    
    # Use PCA for faster initial reduction, then t-SNE if needed
    if descriptors.shape[1] > 50:
        # First reduce with PCA
        pca = PCA(n_components=50)
        descriptors_reduced = pca.fit_transform(descriptors)
        print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")
    else:
        descriptors_reduced = descriptors
    
    # Apply t-SNE for final visualization (only if we have reasonable number of points)
    if len(descriptors_reduced) > 1000:
        print("Using PCA for visualization (too many points for t-SNE)")
        reducer = PCA(n_components=n_components)
    else:
        print("Using t-SNE for visualization...")
        reducer = TSNE(n_components=n_components, random_state=42, perplexity=min(30, len(descriptors_reduced)-1))
    
    descriptors_2d = reducer.fit_transform(descriptors_reduced)
    
    # Create visualization
    fig = plt.figure(figsize=(18, 6))
    
    # Plot 1: Descriptor space (2D or 3D projection)
    if n_components == 2:
        ax1 = fig.add_subplot(131)
        scatter = ax1.scatter(descriptors_2d[:, 0], descriptors_2d[:, 1], 
                             c=range(len(descriptors_2d)), cmap='viridis', 
                             s=20, alpha=0.6)
        ax1.set_xlabel('Component 1')
        ax1.set_ylabel('Component 2')
        ax1.set_title(f'Descriptor Space ({reducer.__class__.__name__})')
        plt.colorbar(scatter, ax=ax1, label='Keypoint Index')
    else:
        ax1 = fig.add_subplot(131, projection='3d')
        scatter = ax1.scatter(descriptors_2d[:, 0], descriptors_2d[:, 1], descriptors_2d[:, 2],
                             c=range(len(descriptors_2d)), cmap='viridis', s=20, alpha=0.6)
        ax1.set_xlabel('Component 1')
        ax1.set_ylabel('Component 2')
        ax1.set_zlabel('Component 3')
        ax1.set_title(f'Descriptor Space ({reducer.__class__.__name__})')
    
    # Plot 2: Image with keypoints colored by descriptor similarity (use first component as color)
    ax2 = fig.add_subplot(132)
    ax2.imshow(img)
    if n_components >= 1:
        colors_1d = descriptors_2d[:, 0]
        scatter2 = ax2.scatter(keypoints[:, 0], keypoints[:, 1], 
                              c=colors_1d, cmap='viridis', s=30, 
                              alpha=0.7, edgecolors='white', linewidths=0.5)
        plt.colorbar(scatter2, ax=ax2, label='Descriptor Component 1')
    ax2.set_title('Keypoints Colored by Descriptor')
    ax2.axis('off')
    
    # Plot 3: All keypoints on image
    ax3 = fig.add_subplot(133)
    ax3.imshow(img)
    ax3.scatter(keypoints[:, 0], keypoints[:, 1], c='red', s=20, alpha=0.6, marker='.')
    ax3.set_title(f'All Detected Keypoints ({len(keypoints)})')
    ax3.axis('off')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize descriptors for image 0
if 'descriptors' in feats0 and 'keypoints' in feats0:
    # Remove batch dimension
    desc0_viz = feats0['descriptors']
    kpts0_desc = feats0['keypoints']
    if desc0_viz.dim() > 2:
        desc0_viz = desc0_viz[0]
        kpts0_desc = kpts0_desc[0]
    
    # Transpose descriptors if needed (should be N x D, not D x N)
    if desc0_viz.shape[0] != len(kpts0_desc):
        desc0_viz = desc0_viz.T
    
    try:
        visualize_descriptors(kpts0_desc, desc0_viz, image0_np, 
                             n_components=2, title="Image 0 Feature Descriptors")
    except Exception as e:
        print(f"Error in descriptor visualization: {e}")
        print("Falling back to simple visualization...")
        visualize_descriptor_stats(kpts0_desc, desc0_viz, image0_np, "Image 0 Feature Descriptors")


In [ ]:
# Visualize matches
def visualize_matches(img0: np.ndarray, img1: np.ndarray, 
                     points0: torch.Tensor, points1: torch.Tensor,
                     mask0: np.ndarray = None, mask1: np.ndarray = None,
                     title: str = "Feature Matches", 
                     line_color: str = 'green',
                     filter_by_mask: bool = False):
    """
    Visualize matched keypoints between two images.
    
    Args:
        img0: First image as numpy array (H, W, 3)
        img1: Second image as numpy array (H, W, 3)
        points0: Keypoints in first image (K, 2) as torch tensor or numpy array
        points1: Keypoints in second image (K, 2) as torch tensor or numpy array
        mask0: Optional binary mask for first image (H, W) - if provided and filter_by_mask=True, only matches within mask are shown
        mask1: Optional binary mask for second image (H, W) - if provided and filter_by_mask=True, only matches within mask are shown
        title: Plot title
        line_color: Color for match lines (default: 'green')
        filter_by_mask: If True and masks are provided, filter matches by mask (default: False, assumes points are already filtered)
    """
    # Convert torch tensors to numpy if needed
    if isinstance(points0, torch.Tensor):
        points0 = points0.cpu().numpy()
    if isinstance(points1, torch.Tensor):
        points1 = points1.cpu().numpy()
    
    # Filter matches by mask if requested and masks are provided
    if filter_by_mask and mask0 is not None and mask1 is not None:
        points0_filtered, points1_filtered = filter_matches_by_mask(points0, points1, mask0, mask1)
        if len(points0_filtered) > 0:
            points0 = points0_filtered
            points1 = points1_filtered
            print(f"Filtered to {len(points0)} matches within mask regions")
        else:
            print("No matches found within mask regions!")
            return
    elif filter_by_mask and (mask0 is not None or mask1 is not None):
        print("Warning: Both masks must be provided for filtering. Showing all matches.")
    
    # Create a side-by-side visualization
    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    h, w = max(h0, h1), w0 + w1
    
    # Create composite image
    composite = np.zeros((h, w, 3), dtype=np.uint8)
    composite[:h0, :w0] = img0
    composite[:h1, w0:w0+w1] = img1
    
    # Adjust points1 x-coordinates to account for offset
    points1_adj = points1.copy()
    points1_adj[:, 0] += w0
    
    # Draw matches
    fig, ax = plt.subplots(figsize=(20, 10))
    ax.imshow(composite)
    ax.set_title(f"{title} ({len(points0)} matches)", fontsize=16)
    ax.axis('off')
    
    # Draw keypoints and lines
    for i in range(len(points0)):
        pt0 = points0[i].astype(int)
        pt1 = points1_adj[i].astype(int)
        
        # Draw line connecting matches (changed to green)
        ax.plot([pt0[0], pt1[0]], [pt0[1], pt1[1]], color=line_color, alpha=0.5, linewidth=1.0)
        
        # Draw keypoints
        ax.plot(pt0[0], pt0[1], 'ro', markersize=4)
        ax.plot(pt1[0], pt1[1], 'bo', markersize=4)  # Changed to blue for better visibility
    
    plt.tight_layout()
    plt.show()

# Visualize the matches (points are already filtered by mask if masks were available)
visualize_matches(image0_np, image1_np, points0, points1, 
                  mask0=mask0, mask1=mask1,
                  title="LightGlue SuperPoint Matches",
                  line_color='green',
                  filter_by_mask=False)  # Don't filter again, already filtered above


In [ ]:
# Visualize pruning information (if available)
# This shows which points were pruned at different layers and when LightGlue stopped
if has_pruning:
    print("Visualizing pruning results...")
    
    # Get all keypoints (not just matched ones)
    kpts0_all = feats0['keypoints']  # All keypoints from image 0
    kpts1_all = feats1['keypoints']  # All keypoints from image 1
    
    # Convert matches to numpy if it's a tensor
    matches_np = matches
    if isinstance(matches, torch.Tensor):
        matches_np = matches.cpu().numpy()
    
    # Convert to numpy if needed
    if isinstance(kpts0_all, torch.Tensor):
        kpts0_all = kpts0_all.cpu().numpy()
    if isinstance(kpts1_all, torch.Tensor):
        kpts1_all = kpts1_all.cpu().numpy()
    
    # Get pruning colors using viz2d utility
    kpc0 = viz2d.cm_prune(matches01['prune0'])
    kpc1 = viz2d.cm_prune(matches01['prune1'])
    
    # Convert images to torch format for viz2d (it expects torch tensors)
    # But we can also use our numpy images with matplotlib
    image0_torch = torch.from_numpy(image0_np).permute(2, 0, 1).float() / 255.0
    image1_torch = torch.from_numpy(image1_np).permute(2, 0, 1).float() / 255.0
    
    # Use viz2d to plot images and keypoints
    axes = viz2d.plot_images([image0_torch, image1_torch])
    
    # Plot matched keypoints with lines (use numpy matches for indexing)
    m_kpts0 = kpts0_all[matches_np[..., 0]]
    m_kpts1 = kpts1_all[matches_np[..., 1]]
    viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=0.2)
    
    # Add text about early stopping if available
    if has_stop:
        viz2d.add_text(0, f'Stop after {matches01["stop"]} layers', fs=20)
    
    plt.show()
    
    # Second plot: Show pruning information with colored keypoints
    # This shows all keypoints colored by when they were pruned
    axes = viz2d.plot_images([image0_torch, image1_torch])
    viz2d.plot_keypoints([kpts0_all, kpts1_all], colors=[kpc0, kpc1], ps=6)
    
    if has_stop:
        viz2d.add_text(0, f'Pruning visualization (stopped after {matches01["stop"]} layers)', fs=20)
    else:
        viz2d.add_text(0, 'Pruning visualization', fs=20)
    
    plt.show()
    
    print("Pruning visualization complete!")
    print("Red points were pruned early, yellow/green points were kept longer, blue points were never pruned")
else:
    print("Pruning information not available in matches01. This may depend on the LightGlue version.")


In [ ]:
# Additional visualization: Show keypoints on individual images
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Image 0 with keypoints
axes[0].imshow(image0_np)
if isinstance(points0, torch.Tensor):
    pts0_np = points0.cpu().numpy()
else:
    pts0_np = points0
axes[0].scatter(pts0_np[:, 0], pts0_np[:, 1], c='red', s=10, alpha=0.6, marker='o')
axes[0].set_title(f"Image 0: {len(pts0_np)} matched keypoints")
axes[0].axis('off')

# Image 1 with keypoints
axes[1].imshow(image1_np)
if isinstance(points1, torch.Tensor):
    pts1_np = points1.cpu().numpy()
else:
    pts1_np = points1
axes[1].scatter(pts1_np[:, 0], pts1_np[:, 1], c='blue', s=10, alpha=0.6, marker='o')
axes[1].set_title(f"Image 1: {len(pts1_np)} matched keypoints")
axes[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Optional: Test with different feature extractors
# Uncomment to try DISK, ALIKED, or SIFT

# # DISK+LightGlue
# extractor_disk = DISK(max_num_keypoints=2048).eval().to(device)
# matcher_disk = LightGlue(features='disk').eval().to(device)
# 
# feats0_disk = extractor_disk.extract(image0)
# feats1_disk = extractor_disk.extract(image1)
# matches01_disk = matcher_disk({'image0': feats0_disk, 'image1': feats1_disk})
# feats0_disk, feats1_disk, matches01_disk = [rbd(x) for x in [feats0_disk, feats1_disk, matches01_disk]]
# matches_disk = matches01_disk['matches']
# points0_disk = feats0_disk['keypoints'][matches_disk[..., 0]]
# points1_disk = feats1_disk['keypoints'][matches_disk[..., 1]]
# 
# print(f"DISK matches: {len(matches_disk)}")
# visualize_matches(image0_np, image1_np, points0_disk, points1_disk, 
#                   title="LightGlue DISK Matches")

print("Feature extraction and matching completed!")


In [ ]:
# Visualize pruning information for the test image pair (if available)
# This cell should be run after the cell that tests different image pairs
# Check if the required variables exist
if 'idx0' in globals() and 'idx1' in globals() and 'matches_new' in globals():
    if idx0 != idx1 and idx0 < len(rgb_images) and idx1 < len(rgb_images):
        has_pruning_new = 'prune0' in matches_new and 'prune1' in matches_new
        has_stop_new = 'stop' in matches_new
        
        if has_pruning_new:
            print("Visualizing pruning results for this image pair...")
            
            # Get all keypoints
            kpts0_all_new = feats0_new['keypoints']
            kpts1_all_new = feats1_new['keypoints']
            
            # Convert matches to numpy if it's a tensor
            matches_new_indices_np = matches_new_indices
            if isinstance(matches_new_indices, torch.Tensor):
                matches_new_indices_np = matches_new_indices.cpu().numpy()
            
            # Convert to numpy if needed
            if isinstance(kpts0_all_new, torch.Tensor):
                kpts0_all_new = kpts0_all_new.cpu().numpy()
            if isinstance(kpts1_all_new, torch.Tensor):
                kpts1_all_new = kpts1_all_new.cpu().numpy()
            
            # Get pruning colors
            kpc0_new = viz2d.cm_prune(matches_new['prune0'])
            kpc1_new = viz2d.cm_prune(matches_new['prune1'])
            
            # Convert images to torch format for viz2d
            img0_torch_new = torch.from_numpy(img0_np).permute(2, 0, 1).float() / 255.0
            img1_torch_new = torch.from_numpy(img1_np).permute(2, 0, 1).float() / 255.0
            
            # Plot matches with pruning info (use numpy matches for indexing)
            axes = viz2d.plot_images([img0_torch_new, img1_torch_new])
            m_kpts0_new = kpts0_all_new[matches_new_indices_np[..., 0]]
            m_kpts1_new = kpts1_all_new[matches_new_indices_np[..., 1]]
            viz2d.plot_matches(m_kpts0_new, m_kpts1_new, color="lime", lw=0.2)
            
            if has_stop_new:
                viz2d.add_text(0, f'Stop after {matches_new["stop"]} layers', fs=20)
            
            plt.show()
            
            # Plot pruning visualization - shows all keypoints colored by pruning layer
            axes = viz2d.plot_images([img0_torch_new, img1_torch_new])
            viz2d.plot_keypoints([kpts0_all_new, kpts1_all_new], colors=[kpc0_new, kpc1_new], ps=6)
            
            if has_stop_new:
                viz2d.add_text(0, f'Pruning: Image {idx0} vs {idx1} (stopped after {matches_new["stop"]} layers)', fs=20)
            else:
                viz2d.add_text(0, f'Pruning: Image {idx0} vs {idx1}', fs=20)
            
            plt.show()
            print("Pruning visualization complete!")
            print("Color legend: Red = pruned early, Yellow/Green = kept longer, Blue = never pruned")
    else:
        print("Invalid image indices or variables not properly set.")
else:
    print("Note: Required variables (idx0, idx1, matches_new) not found. Run the cell that tests different image pairs first.")


In [ ]:
# Alternative approach: Extract masked regions first, then match
# This processes only the masked portion of the image through SuperPoint and LightGlue
print("=" * 60)
print("Approach 2: Extract masked region FIRST, then match")
print("=" * 60)

if 'mask_images' in globals() and len(mask_images) >= 2:
    mask0_crop = mask_images[0]
    mask1_crop = mask_images[1]
    
    # Extract masked regions
    img0_masked, mask0_cropped, bbox0 = extract_masked_region(image0_np, mask0_crop)
    img1_masked, mask1_cropped, bbox1 = extract_masked_region(image1_np, mask1_crop)
    
    print(f"Image 0: Original size {image0_np.shape[:2]}, Masked region size {img0_masked.shape[:2]}")
    print(f"Image 0 bbox: {bbox0}")
    print(f"Image 1: Original size {image1_np.shape[:2]}, Masked region size {img1_masked.shape[:2]}")
    print(f"Image 1 bbox: {bbox1}")
    
    # Display masked regions
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes[0, 0].imshow(image0_np)
    axes[0, 0].set_title("Original Image 0")
    axes[0, 0].axis('off')
    axes[0, 1].imshow(img0_masked)
    axes[0, 1].set_title("Masked Region Image 0")
    axes[0, 1].axis('off')
    axes[1, 0].imshow(image1_np)
    axes[1, 0].set_title("Original Image 1")
    axes[1, 0].axis('off')
    axes[1, 1].imshow(img1_masked)
    axes[1, 1].set_title("Masked Region Image 1")
    axes[1, 1].axis('off')
    plt.tight_layout()
    plt.show()
    
    # Convert masked images to torch tensors
    img0_masked_t = numpy_to_lightglue_image(img0_masked)
    img1_masked_t = numpy_to_lightglue_image(img1_masked)
    
    # Extract features from masked regions
    print("\nExtracting features from masked regions...")
    with torch.no_grad():
        feats0_masked = extractor.extract(img0_masked_t)
        feats1_masked = extractor.extract(img1_masked_t)
        matches_masked = matcher({'image0': feats0_masked, 'image1': feats1_masked})
        feats0_masked, feats1_masked, matches_masked = [rbd(x) for x in [feats0_masked, feats1_masked, matches_masked]]
    
    # Get matches
    matches_masked_indices = matches_masked['matches']
    if isinstance(matches_masked_indices, torch.Tensor):
        matches_masked_indices = matches_masked_indices.cpu().numpy()
    
    # Get keypoints (in cropped image coordinates)
    kpts0_masked = feats0_masked['keypoints'][matches_masked_indices[..., 0]]
    kpts1_masked = feats1_masked['keypoints'][matches_masked_indices[..., 1]]
    
    # Convert to numpy
    if isinstance(kpts0_masked, torch.Tensor):
        kpts0_masked = kpts0_masked.cpu().numpy()
    if isinstance(kpts1_masked, torch.Tensor):
        kpts1_masked = kpts1_masked.cpu().numpy()
    
    print(f"Number of matches in masked regions (before filtering): {len(matches_masked_indices)}")
    
    # Filter matches to ensure they are within the actual mask boundaries (not just the bounding box)
    # Use the cropped masks for filtering
    kpts0_masked_filtered, kpts1_masked_filtered = filter_matches_by_mask(
        kpts0_masked, kpts1_masked, mask0_cropped, mask1_cropped
    )
    
    print(f"Number of matches in masked regions (after mask filtering): {len(kpts0_masked_filtered)}")
    
    # Update keypoints to filtered version
    kpts0_masked = kpts0_masked_filtered
    kpts1_masked = kpts1_masked_filtered
    
    # Transform keypoints back to original image coordinates for visualization
    kpts0_masked_original = transform_keypoints_to_original(kpts0_masked, bbox0)
    kpts1_masked_original = transform_keypoints_to_original(kpts1_masked, bbox1)
    
    # Visualize matches on masked regions (cropped view)
    print("\nVisualizing matches on masked regions (cropped view)...")
    visualize_matches(img0_masked, img1_masked, kpts0_masked, kpts1_masked,
                     title="Matches on Masked Regions (Cropped)",
                     line_color='green',
                     filter_by_mask=False)
    
    # Visualize matches on original images (with coordinate transformation)
    print("\nVisualizing matches on original images (transformed coordinates)...")
    visualize_matches(image0_np, image1_np, kpts0_masked_original, kpts1_masked_original,
                     title="Matches from Masked Regions (Original Image Coordinates)",
                     line_color='green',
                     filter_by_mask=False)
    
    # Compare with the original approach (match first, then filter)
    print("\n" + "=" * 60)
    print("Comparison:")
    print("=" * 60)
    print(f"Approach 1 (match full image, then filter by mask): {len(points0)} matches")
    print(f"Approach 2 (extract masked region, match, then filter by mask): {len(kpts0_masked)} matches")
    print(f"Difference: {len(kpts0_masked) - len(points0)} matches")
    print(f"\nApproach 2 benefits:")
    print(f"  - Processes smaller images (faster)")
    print(f"  - Focuses on region of interest")
    print(f"  - May find more matches in the masked region")
    print(f"  - Still filters to ensure all points are within mask boundaries")
    
    # Side-by-side comparison visualization
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Approach 1: Match then filter
    h0, w0 = image0_np.shape[:2]
    h1, w1 = image1_np.shape[:2]
    h, w = max(h0, h1), w0 + w1
    composite1 = np.zeros((h, w, 3), dtype=np.uint8)
    composite1[:h0, :w0] = image0_np
    composite1[:h1, w0:w0+w1] = image1_np
    
    pts0_comp1 = points0 if isinstance(points0, np.ndarray) else points0.cpu().numpy()
    pts1_comp1 = points1 if isinstance(points1, np.ndarray) else points1.cpu().numpy()
    pts1_comp1_adj = pts1_comp1.copy()
    pts1_comp1_adj[:, 0] += w0
    
    axes[0].imshow(composite1)
    for i in range(len(pts0_comp1)):
        pt0 = pts0_comp1[i].astype(int)
        pt1 = pts1_comp1_adj[i].astype(int)
        axes[0].plot([pt0[0], pt1[0]], [pt0[1], pt1[1]], 'g-', alpha=0.5, linewidth=1.0)
        axes[0].plot(pt0[0], pt0[1], 'ro', markersize=4)
        axes[0].plot(pt1[0], pt1[1], 'bo', markersize=4)
    axes[0].set_title(f"Approach 1: Match Full Image, Then Filter ({len(pts0_comp1)} matches)", fontsize=14)
    axes[0].axis('off')
    
    # Approach 2: Mask first then match
    composite2 = np.zeros((h, w, 3), dtype=np.uint8)
    composite2[:h0, :w0] = image0_np
    composite2[:h1, w0:w0+w1] = image1_np
    
    pts1_comp2_adj = kpts1_masked_original.copy()
    pts1_comp2_adj[:, 0] += w0
    
    axes[1].imshow(composite2)
    for i in range(len(kpts0_masked_original)):
        pt0 = kpts0_masked_original[i].astype(int)
        pt1 = pts1_comp2_adj[i].astype(int)
        axes[1].plot([pt0[0], pt1[0]], [pt0[1], pt1[1]], 'g-', alpha=0.5, linewidth=1.0)
        axes[1].plot(pt0[0], pt0[1], 'ro', markersize=4)
        axes[1].plot(pt1[0], pt1[1], 'bo', markersize=4)
    axes[1].set_title(f"Approach 2: Mask First, Match, Then Filter ({len(kpts0_masked_original)} matches)", fontsize=14)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Masks not available. Skipping masked region extraction approach.")


In [ ]:
# Function to create a frame for video (returns numpy array, not matplotlib figure)
def create_match_frame(img0: np.ndarray, img1: np.ndarray, 
                      points0: np.ndarray, points1: np.ndarray,
                      title: str = "", line_color_bgr: Tuple[int, int, int] = (0, 255, 0)):
    """
    Create a frame showing matches between two images.
    Returns a numpy array (BGR format for OpenCV video writing).
    
    Args:
        img0: First image as numpy array (H, W, 3) RGB
        img1: Second image as numpy array (H, W, 3) RGB
        points0: Keypoints in first image (K, 2)
        points1: Keypoints in second image (K, 2)
        title: Title text to add to frame
        line_color_bgr: Color for match lines in BGR format (default: green)
        
    Returns:
        Frame as numpy array in BGR format
    """
    # Convert to numpy if needed
    if isinstance(points0, torch.Tensor):
        points0 = points0.cpu().numpy()
    if isinstance(points1, torch.Tensor):
        points1 = points1.cpu().numpy()
    
    # Create a side-by-side visualization
    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    h, w = max(h0, h1), w0 + w1
    
    # Create composite image (convert RGB to BGR for OpenCV)
    composite = np.zeros((h, w, 3), dtype=np.uint8)
    composite[:h0, :w0] = cv2.cvtColor(img0, cv2.COLOR_RGB2BGR)
    composite[:h1, w0:w0+w1] = cv2.cvtColor(img1, cv2.COLOR_RGB2BGR)
    
    # Adjust points1 x-coordinates to account for offset
    points1_adj = points1.copy()
    points1_adj[:, 0] += w0
    
    # Draw matches using OpenCV
    for i in range(len(points0)):
        pt0 = points0[i].astype(int)
        pt1 = points1_adj[i].astype(int)
        
        # Draw line connecting matches (BGR format for OpenCV)
        cv2.line(composite, tuple(pt0), tuple(pt1), line_color_bgr, 1, cv2.LINE_AA)
        
        # Draw keypoints
        cv2.circle(composite, tuple(pt0), 4, (0, 0, 255), -1)  # Red for image 0 (BGR)
        cv2.circle(composite, tuple(pt1), 4, (255, 0, 0), -1)  # Blue for image 1 (BGR)
    
    # Add title text
    if title:
        cv2.putText(composite, title, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                   1.0, (255, 255, 255), 2, cv2.LINE_AA)
    
    return composite


In [ ]:
# Create video of matches between consecutive frames
print(f"Processing {len(rgb_images)} images for video creation...")
print("This may take a while depending on the number of images...")

# Video output path
# find parent direction and save to debug/lightglue/
import os

# Get parent directory of the current notebook
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
output_dir = os.path.join(parent_dir, "debug", "lightglue")
os.makedirs(output_dir, exist_ok=True)
output_video_path = os.path.join(output_dir, "lightglue_matches_video.mp4")

# output_video_path = "lightglue_matches_video.mp4"
fps = 2  # Frames per second

# Process all consecutive image pairs
frames = []
total_pairs = len(rgb_images) - 1

for i in range(total_pairs):
    print(f"Processing pair {i}/{total_pairs}: Image {i} -> Image {i+1}")
    
    # Get images
    img0_np = rgb_images[i]
    img1_np = rgb_images[i+1]
    
    # Convert to torch tensors
    img0_t = numpy_to_lightglue_image(img0_np)
    img1_t = numpy_to_lightglue_image(img1_np)
    
    # Extract and match
    with torch.no_grad():
        feats0_vid = extractor.extract(img0_t)
        feats1_vid = extractor.extract(img1_t)
        matches_vid = matcher({'image0': feats0_vid, 'image1': feats1_vid})
        feats0_vid, feats1_vid, matches_vid = [rbd(x) for x in [feats0_vid, feats1_vid, matches_vid]]
    
    # Get matches
    matches_vid_indices = matches_vid['matches']
    if isinstance(matches_vid_indices, torch.Tensor):
        matches_vid_indices = matches_vid_indices.cpu().numpy()
    
    pts0_vid = feats0_vid['keypoints'][matches_vid_indices[..., 0]]
    pts1_vid = feats1_vid['keypoints'][matches_vid_indices[..., 1]]
    
    # Convert to numpy if needed
    if isinstance(pts0_vid, torch.Tensor):
        pts0_vid = pts0_vid.cpu().numpy()
    if isinstance(pts1_vid, torch.Tensor):
        pts1_vid = pts1_vid.cpu().numpy()
    
    # Filter by mask if masks are available
    if 'mask_images' in globals() and len(mask_images) > i+1:
        mask0_vid = mask_images[i]
        mask1_vid = mask_images[i+1]
        if mask0_vid is not None and mask1_vid is not None:
            pts0_vid, pts1_vid = filter_matches_by_mask(pts0_vid, pts1_vid, mask0_vid, mask1_vid)
    
    # Create frame
    title = f"Frame {i} -> {i+1}: {len(pts0_vid)} matches"
    frame = create_match_frame(img0_np, img1_np, pts0_vid, pts1_vid, 
                               title=title, line_color_bgr=(0, 255, 0))  # Green lines (BGR)
    frames.append(frame)

print(f"\nCreated {len(frames)} frames. Writing video to {output_video_path}...")

# Write video
if len(frames) > 0:
    # Get frame dimensions
    h, w = frames[0].shape[:2]
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))
    
    # Write all frames
    for frame in frames:
        out.write(frame)
    
    out.release()
    print(f"Video saved to {output_video_path}")
    print(f"Video specs: {w}x{h} @ {fps} fps, {len(frames)} frames")
else:
    print("No frames to write!")


In [ ]:
# Create video using "mask first, then match" approach
# This processes only the masked regions through SuperPoint and LightGlue
print("=" * 60)
print("Creating video with 'Mask First, Then Match' approach")
print("=" * 60)
print(f"Processing {len(rgb_images)} images for video creation...")
print("This may take a while depending on the number of images...")

if 'mask_images' not in globals() or len(mask_images) == 0:
    print("Masks not available. Skipping masked region video generation.")
else:
    # Video output path
    # Get parent directory of the current notebook
    notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    parent_dir = os.path.dirname(notebook_dir)
    output_dir = os.path.join(parent_dir, "debug", "lightglue")
    os.makedirs(output_dir, exist_ok=True)
    output_video_path_masked = os.path.join(output_dir, "lightglue_matches_video_masked_first.mp4")
    fps = 2  # Frames per second
    
    # Process all consecutive image pairs
    frames_masked = []
    total_pairs = len(rgb_images) - 1
    
    for i in range(total_pairs):
        print(f"Processing pair {i}/{total_pairs}: Image {i} -> Image {i+1}")
        
        # Get images and masks
        img0_np = rgb_images[i]
        img1_np = rgb_images[i+1]
        
        if len(mask_images) > i+1:
            mask0_vid = mask_images[i]
            mask1_vid = mask_images[i+1]
        else:
            print(f"  Warning: Masks not available for pair {i}, skipping...")
            continue
        
        # Extract masked regions
        img0_masked, mask0_cropped, bbox0 = extract_masked_region(img0_np, mask0_vid)
        img1_masked, mask1_cropped, bbox1 = extract_masked_region(img1_np, mask1_vid)
        
        # Convert masked images to torch tensors
        img0_masked_t = numpy_to_lightglue_image(img0_masked)
        img1_masked_t = numpy_to_lightglue_image(img1_masked)
        
        # Extract features from masked regions and match
        with torch.no_grad():
            feats0_masked_vid = extractor.extract(img0_masked_t)
            feats1_masked_vid = extractor.extract(img1_masked_t)
            matches_masked_vid = matcher({'image0': feats0_masked_vid, 'image1': feats1_masked_vid})
            feats0_masked_vid, feats1_masked_vid, matches_masked_vid = [rbd(x) for x in [feats0_masked_vid, feats1_masked_vid, matches_masked_vid]]
        
        # Get matches
        matches_masked_vid_indices = matches_masked_vid['matches']
        if isinstance(matches_masked_vid_indices, torch.Tensor):
            matches_masked_vid_indices = matches_masked_vid_indices.cpu().numpy()
        
        # Get keypoints (in cropped image coordinates)
        kpts0_masked_vid = feats0_masked_vid['keypoints'][matches_masked_vid_indices[..., 0]]
        kpts1_masked_vid = feats1_masked_vid['keypoints'][matches_masked_vid_indices[..., 1]]
        
        # Convert to numpy if needed
        if isinstance(kpts0_masked_vid, torch.Tensor):
            kpts0_masked_vid = kpts0_masked_vid.cpu().numpy()
        if isinstance(kpts1_masked_vid, torch.Tensor):
            kpts1_masked_vid = kpts1_masked_vid.cpu().numpy()
        
        # Filter matches to ensure they are within the actual mask boundaries
        kpts0_masked_vid_filtered, kpts1_masked_vid_filtered = filter_matches_by_mask(
            kpts0_masked_vid, kpts1_masked_vid, mask0_cropped, mask1_cropped
        )
        
        # Transform keypoints back to original image coordinates for visualization
        kpts0_masked_vid_original = transform_keypoints_to_original(kpts0_masked_vid_filtered, bbox0)
        kpts1_masked_vid_original = transform_keypoints_to_original(kpts1_masked_vid_filtered, bbox1)
        
        # Create frame (using original image coordinates for display)
        title = f"Frame {i} -> {i+1}: {len(kpts0_masked_vid_filtered)} matches (masked first)"
        frame = create_match_frame(img0_np, img1_np, kpts0_masked_vid_original, kpts1_masked_vid_original, 
                                   title=title, line_color_bgr=(0, 255, 0))  # Green lines (BGR)
        frames_masked.append(frame)
    
    print(f"\nCreated {len(frames_masked)} frames. Writing video to {output_video_path_masked}...")
    
    # Write video
    if len(frames_masked) > 0:
        # Get frame dimensions
        h, w = frames_masked[0].shape[:2]
        
        # Create video writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path_masked, fourcc, fps, (w, h))
        
        # Write all frames
        for frame in frames_masked:
            out.write(frame)
        
        out.release()
        print(f"Video saved to {output_video_path_masked}")
        print(f"Video specs: {w}x{h} @ {fps} fps, {len(frames_masked)} frames")
        print("\nComparison:")
        print(f"  Approach 1 video: lightglue_matches_video.mp4 (match full image, then filter)")
        print(f"  Approach 2 video: {output_video_path_masked} (mask first, then match, then filter)")
    else:
        print("No frames to write!")


In [ ]:
# Test with different image pairs
# You can modify these indices to test different image pairs
idx0 = 0
idx1 = min(1, len(rgb_images) - 1)  # Use second image, or last if only one image

if idx0 != idx1 and idx0 < len(rgb_images) and idx1 < len(rgb_images):
    print(f"Testing with image pair: {idx0} and {idx1}")
    
    # Convert to torch tensors
    img0_np = rgb_images[idx0]
    img1_np = rgb_images[idx1]
    
    img0_t = numpy_to_lightglue_image(img0_np)
    img1_t = numpy_to_lightglue_image(img1_np)
    
    # Extract and match
    with torch.no_grad():
        feats0_new = extractor.extract(img0_t)
        feats1_new = extractor.extract(img1_t)
        matches_new = matcher({'image0': feats0_new, 'image1': feats1_new})
        feats0_new, feats1_new, matches_new = [rbd(x) for x in [feats0_new, feats1_new, matches_new]]
    
    matches_new_indices = matches_new['matches']
    pts0_new = feats0_new['keypoints'][matches_new_indices[..., 0]]
    pts1_new = feats1_new['keypoints'][matches_new_indices[..., 1]]
    
    print(f"Number of matches (before mask filtering): {len(matches_new_indices)}")
    
    # Get masks if available
    mask0_new = mask_images[idx0] if len(mask_images) > idx0 else None
    mask1_new = mask_images[idx1] if len(mask_images) > idx1 else None
    
    # Filter matches by mask if masks are available
    if mask0_new is not None and mask1_new is not None:
        pts0_new_filtered, pts1_new_filtered = filter_matches_by_mask(pts0_new, pts1_new, mask0_new, mask1_new)
        print(f"Number of matches (after mask filtering): {len(pts0_new_filtered)}")
        pts0_new = pts0_new_filtered
        pts1_new = pts1_new_filtered
    
    # Display match statistics
    if 'matching_scores' in matches_new:
        scores = matches_new['matching_scores']
        print(f"Average matching score: {scores.mean().item():.4f}")
        print(f"Min matching score: {scores.min().item():.4f}")
        print(f"Max matching score: {scores.max().item():.4f}")
    
    # Visualize with mask filtering (points are already filtered if masks were available)
    visualize_matches(img0_np, img1_np, pts0_new, pts1_new,
                      mask0=mask0_new, mask1=mask1_new,
                      title=f"LightGlue Matches: Image {idx0} vs Image {idx1}",
                      line_color='green',
                      filter_by_mask=False)  # Don't filter again, already filtered above
else:
    print(f"Cannot test with indices {idx0} and {idx1}")
